In [1]:
import os
import pathlib
import timeit
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
TFLITE_DIR = pathlib.Path("../models/tflite")
NOTEBOOK_DIR = pathlib.Path(".")

mlp_tflite_models = {
    "MLP_Original_baseline":   ("MLP_Original_baseline.tflite", False),
    "MLP_Original_quant":      ("MLP_Original_quant.tflite", True),
    "MLP_Node_baseline":       ("MLP_Node_baseline.tflite", False),
    "MLP_Node_quant":          ("MLP_Node_quant.tflite", True),
    "MLP_Weight_baseline":     ("MLP_Weight_baseline.tflite", False),
    "MLP_Weight_quant":        ("MLP_Weight_quant.tflite", True),
    "MLP_WeightNode_baseline": ("MLP_WeightNode_baseline.tflite", False),
    "MLP_WeightNode_quant":    ("MLP_WeightNode_quant.tflite", True),
}

print("Models found:")
for name, (filename, _) in mlp_tflite_models.items():
    path = TFLITE_DIR / filename
    print(f"  {name}: {'OK' if path.exists() else 'NOT FOUND'} — {path}")

Models found:
  MLP_Original_baseline: OK — ..\models\tflite\MLP_Original_baseline.tflite
  MLP_Original_quant: OK — ..\models\tflite\MLP_Original_quant.tflite
  MLP_Node_baseline: OK — ..\models\tflite\MLP_Node_baseline.tflite
  MLP_Node_quant: OK — ..\models\tflite\MLP_Node_quant.tflite
  MLP_Weight_baseline: OK — ..\models\tflite\MLP_Weight_baseline.tflite
  MLP_Weight_quant: OK — ..\models\tflite\MLP_Weight_quant.tflite
  MLP_WeightNode_baseline: OK — ..\models\tflite\MLP_WeightNode_baseline.tflite
  MLP_WeightNode_quant: OK — ..\models\tflite\MLP_WeightNode_quant.tflite


In [13]:
data = np.load('../data/fdia_dataset_processed.npz')
X_test = data['X_test']       # (N, 6, 83) — mantém assim, sem achatar
y_test = data['y_test']

print(f"X_test: {X_test.shape}")

X_test: (9720, 6, 83)


In [10]:
def measure_batch_inference_tflite(tflite_path, X_test, quantized,
                                    timing_fraction=0.25, n_repeats=5, seed=42):
    interpreter = tf.lite.Interpreter(
        model_path=str(tflite_path),
        num_threads=1,
        experimental_delegates=[]
    )
    interpreter.allocate_tensors()
    in_d = interpreter.get_input_details()[0]
    out_d = interpreter.get_output_details()[0]

    if quantized:
        input_scale, input_zero_point = in_d['quantization']

    rng = np.random.default_rng(seed)
    sample_size = int(timing_fraction * len(X_test))

    batch_times = []

    for _ in range(n_repeats):
        idx = rng.choice(len(X_test), size=sample_size, replace=False)
        X_batch = X_test[idx].astype(np.float32)

        if quantized:
            X_batch = (X_batch / input_scale + input_zero_point).astype(np.uint8)

        start = timeit.default_timer()
        for sample in X_batch:
            interpreter.set_tensor(in_d['index'], np.expand_dims(sample, axis=0))
            interpreter.invoke()
            _ = interpreter.get_tensor(out_d['index'])
        end = timeit.default_timer()

        elapsed_ms = (end - start) * 1000
        per_sample_ms = elapsed_ms / sample_size
        batch_times.append(per_sample_ms)

    del interpreter
    return np.mean(batch_times), batch_times

In [14]:
results = []

for name, (filename, quantized) in mlp_tflite_models.items():
    path = TFLITE_DIR / filename

    avg_time, all_times = measure_batch_inference_tflite(
        path, X_test, quantized, timing_fraction=0.25, n_repeats=5   # ← X_test, não X_test_flat
    )
    

    print(f"{name}: {avg_time:.4f} ms/sample (avg of {len(all_times)} runs) | runs: {[f'{t:.4f}' for t in all_times]}")

    results.append({
        "model": name,
        "batch_inference_time_ms_per_sample": avg_time,
    })

df_batch_tflite = pd.DataFrame(results)
df_batch_tflite.to_csv(NOTEBOOK_DIR / "MLP_tflite_batch_inference_results.csv", index=False)
df_batch_tflite

MLP_Original_baseline: 0.0165 ms/sample (avg of 5 runs) | runs: ['0.0180', '0.0142', '0.0192', '0.0143', '0.0167']
MLP_Original_quant: 0.0152 ms/sample (avg of 5 runs) | runs: ['0.0176', '0.0141', '0.0141', '0.0140', '0.0163']
MLP_Node_baseline: 0.0148 ms/sample (avg of 5 runs) | runs: ['0.0163', '0.0139', '0.0116', '0.0149', '0.0171']
MLP_Node_quant: 0.0120 ms/sample (avg of 5 runs) | runs: ['0.0143', '0.0115', '0.0112', '0.0115', '0.0113']
MLP_Weight_baseline: 0.0150 ms/sample (avg of 5 runs) | runs: ['0.0141', '0.0159', '0.0158', '0.0143', '0.0149']
MLP_Weight_quant: 0.0135 ms/sample (avg of 5 runs) | runs: ['0.0144', '0.0129', '0.0138', '0.0133', '0.0131']
MLP_WeightNode_baseline: 0.0171 ms/sample (avg of 5 runs) | runs: ['0.0153', '0.0157', '0.0171', '0.0191', '0.0183']
MLP_WeightNode_quant: 0.0128 ms/sample (avg of 5 runs) | runs: ['0.0133', '0.0125', '0.0127', '0.0125', '0.0130']


,model,batch_inference_time_ms_per_sample
0,MLP_Original_baseline,0.016488
1,MLP_Original_quant,0.015244
2,MLP_Node_baseline,0.014767
3,MLP_Node_quant,0.011980
4,MLP_Weight_baseline,0.014993
5,MLP_Weight_quant,0.013489
6,MLP_WeightNode_baseline,0.017111
7,MLP_WeightNode_quant,0.012820
